In [1]:
ls

 Volume in drive C is OS
 Volume Serial Number is 9215-BD69

 Directory of C:\Stuff\M.Tech\Data_Engg\projet\code

28-10-2026  20:03    <DIR>          .
28-10-2026  13:14    <DIR>          ..
28-10-2026  15:03    <DIR>          .ipynb_checkpoints
28-10-2026  20:03            27,898 data_engg.ipynb
28-10-2026  11:47    <DIR>          dataset
26-10-2026  16:10    <DIR>          etc
26-10-2026  16:00    <DIR>          Include
26-10-2026  16:00    <DIR>          Lib
15-10-2026  17:58         1,950,748 links.csv
15-10-2026  17:58         4,242,926 movies.csv
26-10-2026  16:00               196 pyvenv.cfg
15-10-2026  17:58       877,076,222 ratings.csv
26-10-2026  16:10    <DIR>          Scripts
26-10-2026  16:10    <DIR>          share
15-10-2026  17:58        72,353,890 tags.csv
28-10-2026  15:02            29,692 Untitled.ipynb
               7 File(s)    955,681,572 bytes
               9 Dir(s)  204,851,515,392 bytes free


In [54]:
import io
import time
import re
import os
import numpy as np
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [46]:
print("Loading raw datasets...")

raw_movies = pd.read_csv('movies.csv', dtype={'movieId': 'int32', 'title': 'str', 'genres': 'str'})
raw_links  = pd.read_csv('links.csv', dtype={'movieId': 'int32', 'imdbId': 'int32', 'tmdbId': 'Int32'})
raw_ratings = pd.read_csv('ratings.csv', dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int64'})
raw_tags    = pd.read_csv('tags.csv', dtype={'userId': 'int32', 'movieId': 'int32', 'tag': 'str', 'timestamp': 'int64'})

Loading raw datasets...


In [17]:
raw_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [18]:
raw_tags.head()

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [19]:
raw_links.head()

,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


In [20]:
raw_ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [21]:
raw_movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  87585 non-null  int32
 1   title    87585 non-null  str  
 2   genres   87585 non-null  str  
dtypes: int32(1), str(2)
memory usage: 1.7 MB


In [9]:
raw_ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int32  
 1   movieId    int32  
 2   rating     float32
 3   timestamp  int64  
dtypes: float32(1), int32(2), int64(1)
memory usage: 610.4 MB


In [10]:
raw_links.info()

<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  87585 non-null  int32
 1   imdbId   87585 non-null  int32
 2   tmdbId   87461 non-null  Int32
dtypes: Int32(1), int32(2)
memory usage: 1.1 MB


In [22]:
raw_tags.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000072 entries, 0 to 2000071
Data columns (total 4 columns):
 #   Column     Dtype
---  ------     -----
 0   userId     int32
 1   movieId    int32
 2   tag        str  
 3   timestamp  int64
dtypes: int32(2), int64(1), str(1)
memory usage: 45.8 MB


### EDA

In [ ]:
ord_columns = [
    
]

# collecting all unique genres and tags

genres_list = [
    col.lower() for col in movies['genres']
]

genres = []

for i in genres_list:
    elems = re.split("|[a-z]|", i)
    genres.append(elems)

genres_list[3]

In [ ]:
raw_tags[raw_tags['movieId']==17]

### Cleaning the titles and rearranging the tables to fit OLAP structure

In [23]:
print("Building dim_movies...")

dim_movies = raw_movies.copy()

dim_movies['release_year'] = dim_movies['title'].str.extract(r'\((\d{4})\)$').astype('Int32')
dim_movies['clean_title'] = dim_movies['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True).str.strip()
dim_movies = dim_movies[['movieId', 'clean_title', 'release_year']].merge(raw_links, on='movieId', how='left')

Building dim_movies...


### Normalize the tables - 1NF

In [24]:
exploded_genres = raw_movies[['movieId', 'genres']].copy()
exploded_genres['genre_name'] = exploded_genres['genres'].str.split('|')
exploded_genres = exploded_genres.explode('genre_name')

# Extract unique genre dimension (dim_genres)
unique_genres = pd.DataFrame({'genre_name': sorted(exploded_genres['genre_name'].dropna().unique())})
unique_genres['genreId'] = np.arange(1, len(unique_genres) + 1, dtype='int16')
dim_genres = unique_genres[['genreId', 'genre_name']]

# Create Bridge Table (bridge_movie_genre)
bridge_movie_genre = exploded_genres.merge(dim_genres, on='genre_name', how='inner')
bridge_movie_genre = bridge_movie_genre[['movieId', 'genreId']].drop_duplicates()

### building dimension users - this separates the numerical/facts from the attributes/dimension which are used for analysis. Connecting the two makes for an OLAP structure, optimised for querying

In [25]:
user_stats = raw_ratings.groupby('userId').agg(
    total_ratings_given=('rating', 'count'),
    avg_rating_given=('rating', 'mean'),
    first_active_ts=('timestamp', 'min'),
    last_active_ts=('timestamp', 'max')
).reset_index()

user_stats['first_active_dt'] = pd.to_datetime(user_stats['first_active_ts'], unit='s')
user_stats['last_active_dt']  = pd.to_datetime(user_stats['last_active_ts'], unit='s')

dim_users = user_stats.drop(columns=['first_active_ts', 'last_active_ts'])

### building dimension date to handle all the time-series data -timestamps of user ratings and tags

In [48]:
# Convert timestamps to datetime in raw_ratings & raw_tags
raw_ratings['datetime'] = pd.to_datetime(raw_ratings['timestamp'], unit='s')
raw_tags['datetime']    = pd.to_datetime(raw_tags['timestamp'], unit='s')

# Extract unique dates from ratings and tags
unique_dates = pd.Series(
    pd.concat([raw_ratings['datetime'].dt.date, raw_tags['datetime'].dt.date]).unique()
).sort_values().reset_index(drop=True)

dim_date = pd.DataFrame({'full_date': pd.to_datetime(unique_dates)})
dim_date['dateKey']     = dim_date['full_date'].dt.strftime('%Y%m%d').astype('int32')
dim_date['year']        = dim_date['full_date'].dt.year.astype('int16')
dim_date['quarter']     = dim_date['full_date'].dt.quarter.astype('int8')
dim_date['month']       = dim_date['full_date'].dt.month.astype('int8')
dim_date['day_of_week'] = dim_date['full_date'].dt.day_of_week.astype('int8')

### building fact tables - ratings and tags

In [49]:
fact_ratings = raw_ratings.copy()
fact_ratings['dateKey'] = fact_ratings['datetime'].dt.strftime('%Y%m%d').astype('int32')
fact_ratings['ratingId'] = np.arange(1, len(fact_ratings) + 1, dtype='int64')
fact_ratings = fact_ratings[['ratingId', 'userId', 'movieId', 'dateKey', 'rating', 'datetime']]

fact_tags = raw_tags.copy()
fact_tags['tag_clean'] = fact_tags['tag'].str.lower().str.strip()
fact_tags['dateKey']   = fact_tags['datetime'].dt.strftime('%Y%m%d').astype('int32')
fact_tags['tagId']     = np.arange(1, len(fact_tags) + 1, dtype='int64')
fact_tags = fact_tags[['tagId', 'userId', 'movieId', 'dateKey', 'tag_clean', 'datetime']]

### Testing the new tables and data

### connecting to DB

In [57]:
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    db_user = input("PostgreSQL username: ")
    db_password = getpass("PostgreSQL password: ")
    db_host = input("Host [localhost]: ").strip() or "localhost"
    db_port = input("Port [5432]: ").strip() or "5432"
    DATABASE_URL = URL.create(
        "postgresql", username=db_user, password=db_password,
        host=db_host, port=int(db_port), database="movie_lens"
    )
engine = create_engine(DATABASE_URL, echo=False, pool_pre_ping=True)
print("Engine created.")

PostgreSQL username:  gumbo
PostgreSQL password:  ········
Host [localhost]:  localhost
Port [5432]:  5432


Engine created.


### exporting the data

In [ ]:
print("Starting data ingestion into PostgreSQL...")

print("Loading dim_genres...")
dim_genres.rename(columns={
    'genreId': 'genre_id', 
    'genre_name': 'genre_name'
})[['genre_id', 'genre_name']].to_sql(
    'dim_genres', con=engine, if_exists='append', index=False
)

print("Loading dim_movies...")
dim_movies.rename(columns={
    'movieId': 'movie_id', 
    'clean_title': 'title', 
    'release_year': 'release_year'
})[['movie_id', 'title', 'release_year']].to_sql(
    'dim_movies', con=engine, if_exists='append', index=False, chunksize=10000
)

print("Loading bridge_movie_genre...")
bridge_movie_genre.rename(columns={
    'movieId': 'movie_id', 
    'genreId': 'genre_id'
})[['movie_id', 'genre_id']].to_sql(
    'bridge_movie_genre', con=engine, if_exists='append', index=False, chunksize=20000
)

print("Loading dim_users...")
dim_users.rename(columns={
    'userId': 'user_id'
})[['user_id']].to_sql(
    'dim_users', con=engine, if_exists='append', index=False, chunksize=25000
)

print("Loading dim_date...")
dim_date.rename(columns={
    'dateKey': 'date_key', 
    'full_date': 'full_date', 
    'year': 'year', 
    'quarter': 'quarter', 
    'month': 'month'
})[['date_key', 'full_date', 'year', 'quarter', 'month']].to_sql(
    'dim_date', con=engine, if_exists='append', index=False
)

### TAKES TOO LONG

In [ ]:
print("Loading fact_ratings (32M rows)...")
fact_ratings.rename(columns={
    'userId': 'user_id',
    'movieId': 'movie_id',
    'dateKey': 'date_key'
})[['user_id', 'movie_id', 'date_key', 'rating']].to_sql(
    'fact_ratings', con=engine, if_exists='append', index=False, chunksize=10000, method='multi'
)

In [61]:
print("Loading fact_tags...")
fact_tags.rename(columns={
    'userId': 'user_id',
    'movieId': 'movie_id',
    'dateKey': 'date_key',
    'tag_clean': 'tag'
})[['user_id', 'movie_id', 'date_key', 'tag']].to_sql(
    'fact_tags', con=engine, if_exists='append', index=False, chunksize=50000
)

print("All tables successfully loaded into PostgreSQL!")

Loading fact_tags...


error ignored terminating <psycopg.Pipeline [INERROR, pipeline=ON] (host=localhost user=gumbo database=movie_lens) at 0x20d48c02690>: pipeline aborted


DatabaseError: Execution failed on sql 'INSERT INTO fact_tags (user_id, movie_id, date_key, tag) VALUES (:user_id, :movie_id, :date_key, :tag)': (psycopg.errors.NotNullViolation) null value in column "tag" of relation "fact_tags" violates not-null constraint
DETAIL:  Failing row contains (185378, 27046, 33826, 20080915, null).
[SQL: INSERT INTO fact_tags (user_id, movie_id, date_key, tag) VALUES (%(user_id)s::INTEGER, %(movie_id)s::INTEGER, %(date_key)s::INTEGER, %(tag)s::VARCHAR)]
[parameters: [{'user_id': 21942, 'movie_id': 57669, 'date_key': 20090710, 'tag': 'imdb top 250'}, {'user_id': 21942, 'movie_id': 57669, 'date_key': 20090710, 'tag': 'irish accent'}, {'user_id': 21942, 'movie_id': 58301, 'date_key': 20100902, 'tag': 'tim roth'}, {'user_id': 21942, 'movie_id': 58301, 'date_key': 20100902, 'tag': 'torture'}, {'user_id': 21942, 'movie_id': 58306, 'date_key': 20100217, 'tag': 'fewer than 300 ratings'}, {'user_id': 21942, 'movie_id': 59369, 'date_key': 20090624, 'tag': 'cynical'},

### Trying a new method

In [60]:
print("Starting high-speed COPY ingestion for fact_ratings (Psycopg 3)...")
start_time = time.time()

# 1. Prepare the buffer as a tab-separated CSV stream in memory
buffer = io.StringIO()
fact_ratings.rename(columns={
    'userId': 'user_id',
    'movieId': 'movie_id',
    'dateKey': 'date_key'
})[['user_id', 'movie_id', 'date_key', 'rating']].to_csv(
    buffer, sep='\t', index=False, header=False, na_rep='\\N'
)
buffer.seek(0)

# 2. Stream directly into PostgreSQL using Psycopg 3's native copy API
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cursor:
        with cursor.copy(
            "COPY fact_ratings (user_id, movie_id, date_key, rating) FROM STDIN WITH CSV DELIMITER E'\t'"
        ) as copym:
            copym.write(buffer.read())
    raw_conn.commit()
    print(f"Successfully loaded 32M rows in {time.time() - start_time:.2f} seconds!")
except Exception as e:
    raw_conn.rollback()
    print(f"Error during copy: {e}")
finally:
    raw_conn.close()

Starting high-speed COPY ingestion for fact_ratings (Psycopg 3)...
Successfully loaded 32M rows in 358.73 seconds!


In [62]:
print("Starting high-speed COPY ingestion for fact_tags...")
start_time = time.time()

# 1. Prepare the buffer as a tab-separated CSV stream in memory
buffer = io.StringIO()
fact_tags.rename(columns={
    'userId': 'user_id',
    'movieId': 'movie_id',
    'dateKey': 'date_key',
    'tag_clean': 'tag'
})[['user_id', 'movie_id', 'date_key', 'tag']].to_csv(
    buffer, sep='\t', index=False, header=False, na_rep='\\N'
)
buffer.seek(0)

# 2. Stream directly into PostgreSQL using Psycopg 3's native copy API
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cursor:
        with cursor.copy(
            "COPY fact_tags (user_id, movie_id, date_key, tag) FROM STDIN WITH CSV DELIMITER E'\t'"
        ) as copym:
            copym.write(buffer.read())
    raw_conn.commit()
    print(f"Successfully loaded fact_tags in {time.time() - start_time:.2f} seconds!")
except Exception as e:
    raw_conn.rollback()
    print(f"Error during copy: {e}")
finally:
    raw_conn.close()

Starting high-speed COPY ingestion for fact_tags...
Error during copy: null value in column "tag" of relation "fact_tags" violates not-null constraint
DETAIL:  Failing row contains (201002, 2557, 32657, 20090311, null).
CONTEXT:  COPY fact_tags, line 15624: "2557	32657	20090311	"


In [64]:
# handle null values

print("Starting high-speed COPY ingestion for fact_tags...")
start_time = time.time()

# 1. Rename columns
df_tags_prep = fact_tags.rename(columns={
    'userId': 'user_id',
    'movieId': 'movie_id',
    'dateKey': 'date_key',
    'tag_clean': 'tag'
})[['user_id', 'movie_id', 'date_key', 'tag']]

# 2. Aggressively clean missing and empty/whitespace-only tags
df_tags_prep['tag'] = df_tags_prep['tag'].astype(str).str.strip()
df_tags_clean = df_tags_prep.dropna(subset=['tag'])
df_tags_clean = df_tags_clean[
    (df_tags_clean['tag'] != '') & 
    (df_tags_clean['tag'].str.lower() != 'nan') & 
    (df_tags_clean['tag'].str.lower() != 'none')
]

# 3. Prepare the buffer as a tab-separated CSV stream in memory
buffer = io.StringIO()
df_tags_clean.to_csv(
    buffer, sep='\t', index=False, header=False, na_rep='\\N'
)
buffer.seek(0)

# 4. Stream directly into PostgreSQL using Psycopg 3's native copy API
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cursor:
        with cursor.copy(
            "COPY fact_tags (user_id, movie_id, date_key, tag) FROM STDIN WITH CSV DELIMITER E'\t'"
        ) as copym:
            copym.write(buffer.read())
    raw_conn.commit()
    print(f"Successfully loaded fact_tags in {time.time() - start_time:.2f} seconds!")
except Exception as e:
    raw_conn.rollback()
    print(f"Error during copy: {e}")
finally:
    raw_conn.close()

Starting high-speed COPY ingestion for fact_tags...
Successfully loaded fact_tags in 87.00 seconds!
